In [ ]:
from ase.io import read
from ase.visualize import view

import nqetools as nqe

In [ ]:
# Make a directory to store everything
directory_opti = "opti"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"

directory_md_d = "md"
directory_meta_md_d = "meta_md"
directory_meta_pimd_d = "meta_pimd"

tol_energy = 1.0e-3
tol_force = 1.0e-3
tol_position = 1.0e-3

n_beads = 4
timestep = 1.0  # fs
total_steps_md = 100
total_steps_plumed = 5_000 # 20_000

fix_com = True

stride = 10
temperature = 300.0
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'ase-mace'

# Expensive settings
driver_args = {'model': 'large',
               'device': 'cuda',
               'default_dtype': 'float64'}

# Medium settings
driver_args = {'model': 'medium',
               'device': 'cuda',
               'default_dtype': 'float32'}

# Cheap settings
driver_args = {'model': 'small',
               'device': 'cuda',
               'default_dtype': 'float32'}

# Plumed hills settings
n_bins = 100
stride_hills = 100

# # Plumed settings
plumed_type_opes = "opes-diff1"
plumed_args_opes = {'idx1': 1,
                    'idx2': 8,
                    'idx3': 0,
                    'barrier': 0.1,
                    'stride_hills': stride_hills}

# plumed_type_opes = "opes-dist"
# plumed_args_opes = {'idx1': 1,
#                     'idx2': 8,
#                     'barrier': 0.5,
#                     'stride_hills': stride_hills}

plumed_type_opes = "opes_com"
plumed_args_opes = {'group_1': [5, 6, 7, 8, 9],
                    'group_2': [0, 1, 2, 3, 4],
                    'barrier': 0.1,
                    'stride_hills': stride_hills}

plumed_type_opes = "opes_1pt"
plumed_args_opes = {'idx_d': 0,
                    'idx_h': 4,
                    'idx_a': 6,
                    'barrier': 0.5,
                    'stride_hills': stride_hills}
dim = 1

plumed_type_opes = "opes_2pt_2d_coord"
plumed_args_opes = {'idx_d1': 5,
                    'idx_h1': 30,
                    'idx_a1': 21,
                    'idx_d2': 19,
                    'idx_h2': 13,
                    'idx_a2': 6,
                    'barrier': 1.0,
                    'stride_hills': stride_hills}
dim = 2

plumed_type_opes = "opes_2pt_1d_coord"
plumed_args_opes = {'idx_d1': 5,
                    'idx_h1': 30,
                    'idx_a1': 21,
                    'idx_d2': 19,
                    'idx_h2': 13,
                    'idx_a2': 6,
                    'barrier': 0.7,
                    'stride_hills': stride_hills,
                    'explore': False}
dim = 1

# plumed_type_opes = "opes_2pt_1d_coord_com"
# plumed_args_opes = {'idx_d1': 5,
#                     'idx_h1': 30,
#                     'idx_a1': 21,
#                     'idx_d2': 19,
#                     'idx_h2': 13,
#                     'idx_a2': 6,
#                     'group_1': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
#                     'group_2': [16, 17, 18, 19, 20, 21, 22, 23, 24],
#                     'barrier': 0.7,
#                     'stride_hills': stride_hills,
#                     'explore': False}
# dim = 2


plumed_type_opes = "opes_1pt_3donor_coord"
plumed_args_opes = {'idx_d1': 5,
                    'idx_d2': 21,
                    'idx_d3': 19,
                    'idx_h': 30,
                    'barrier': 2.0,
                    'stride_hills': stride_hills}
dim = 1

if dim == 1:
    cv_limits = [None, None]
elif dim == 2:
    cv_limits = [[None, None], [None, None]]

In [ ]:
# Make the system
atoms = read('neb.traj', index='-1') # 0
atoms.center(vacuum=10.0)
view(atoms)

In [ ]:
# Run minimisation
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code,
                          driver_args=driver_args,
                          tol_energy=tol_energy,
                          tol_force=tol_force,
                          tol_position=tol_position)
atoms_opti, output_data_opti, output_desc_opti = output
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti, save=False)

In [ ]:
view(atoms_opti)

In [ ]:
# Run unbiased MD
output = nqe.run_md(directory_md,
                    atoms_opti,
                    driver=driver_code,
                    driver_args=driver_args,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    fix_com=fix_com,
                    stride=1,
                    n_beads=1)
atoms_md, output_data_md, output_desc_md = output

In [ ]:
view(atoms_md)

In [ ]:
# Run OPES metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps_plumed,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
view(atoms_meta_md)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_potential_bias(output_data_meta_md, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_md,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Plot the free energy surface convergence
fes_arrays_meta_md = nqe.load_fes_data(directory_meta_md, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps_plumed, fes_arrays_meta_md)
if dim == 1:
    nqe.plot_fes_series_1d(fes_arrays_meta_md, fes_times, save=False)
else:
    nqe.plot_fes_contourf_series(fes_arrays_meta_md, fes_times, save=False)

In [ ]:
# Run PIMD OPES metadynamics
output = nqe.run_plumed_md(directory_meta_pimd,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps_plumed,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_pimd, output_data_meta_pimd, output_desc_meta_pimd = output

In [ ]:
view(atoms_meta_pimd)

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd, save=False)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd, save=False)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_pimd,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Plot the free energy surface convergence
fes_arrays_meta_pimd = nqe.load_fes_data(directory_meta_pimd, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps_plumed, fes_arrays_meta_pimd)

if dim == 1:
    nqe.plot_fes_series_1d(fes_arrays_meta_pimd, fes_times, save=False)
else:
    nqe.plot_fes_contourf_series(fes_arrays_meta_pimd, fes_times, save=False)


In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
if dim == 1:
    nqe.plot_fes_series_1d_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1], save=False)
else:
    nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1], save=False)

In [ ]:
# Run unbiased deuterated MD
output = nqe.run_md(directory_md_d,
                    atoms_opti,
                    driver=driver_code,
                    driver_args=driver_args,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    fix_com=fix_com,
                    deuterate=True,
                    stride=1,
                    n_beads=1)
atoms_md_d, output_data_md_d, output_desc_md_d = output

In [ ]:
# Run OPES deuterated metadynamics
output = nqe.run_plumed_md(directory_meta_md_d,
                           atoms_md,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps_plumed,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           deuterate=True,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md_d, output_data_meta_md_d, output_desc_meta_md_d = output
nqe.plot_time_potential_bias(output_data_meta_md_d, save=False)
nqe.plot_time_potential_bias(output_data_meta_md_d, save=False)
nqe.plot_time_temperature(output_data_meta_md_d, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_md_d,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Plot the free energy surface convergence
fes_arrays_meta_md_d = nqe.load_fes_data(directory_meta_md_d, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps_plumed, fes_arrays_meta_md_d)
if dim == 1:
    nqe.plot_fes_series_1d(fes_arrays_meta_md_d, fes_times, save=False)
else:
    nqe.plot_fes_contourf_series(fes_arrays_meta_md_d, fes_times, save=False)

# Plot the comparison between MD and PIMD, compare the last frame
if dim == 1:
    nqe.plot_fes_series_1d_compare(fes_arrays_meta_md[-1], fes_arrays_meta_md_d[-1], save=False, labels=['H', 'D'])
else:
    nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_md_d[-1], save=False, labels=['H', 'D'])

In [ ]:
# Run PIMD OPES deuterated metadynamics
output = nqe.run_plumed_md(directory_meta_pimd_d,
                           atoms_md_d,
                           driver=driver_code,
                           driver_args=driver_args,
                           total_steps=total_steps_plumed,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           fix_com=fix_com,
                           deuterate=True,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_pimd_d, output_data_meta_pimd_d, output_desc_meta_pimd_d = output

# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd_d, save=False)
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd_d, save=False)
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd_d, save=False)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_pimd_d,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)
# Plot the free energy surface convergence
fes_arrays_meta_pimd_d = nqe.load_fes_data(directory_meta_pimd_d, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps_plumed, fes_arrays_meta_pimd_d)

if dim == 1:
    nqe.plot_fes_series_1d(fes_arrays_meta_pimd_d, fes_times, save=False)
else:
    nqe.plot_fes_contourf_series(fes_arrays_meta_pimd_d, fes_times, save=False)

# Plot the comparison between MD and PIMD, compare the last frame
if dim == 1:
    nqe.plot_fes_series_1d_compare(fes_arrays_meta_pimd[-1], fes_arrays_meta_pimd_d[-1], save=False, labels=['H', 'D'])
else:
    nqe.plot_fes_contourf_compare(fes_arrays_meta_pimd[-1], fes_arrays_meta_pimd_d[-1], save=False, labels=['H', 'D'])

In [ ]:
nqe.remove_directory(directory_opti)
nqe.remove_directory(directory_md)
nqe.remove_directory(directory_meta_md)
nqe.remove_directory(directory_meta_pimd)

nqe.remove_directory(directory_md_d)
nqe.remove_directory(directory_meta_md_d)
nqe.remove_directory(directory_meta_pimd_d)